# Milestone 4

## Task 1 

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../data/bug_submissions.db")
df = pd.read_sql("SELECT * FROM bug_submissions", conn)
conn.close()

print("Total bugs in database:", len(df))
print("\nColumns:", df.columns.tolist())
print("\nSeverity breakdown:")
print(df['severity'].value_counts())
print("\nComponent breakdown:")
print(df['component'].value_counts())
print("\nError type breakdown:")
print(df['error_type'].value_counts())

Total bugs in database: 37

Columns: ['bug_id', 'severity', 'priority', 'component', 'error_type', 'failure_location', 'code_path', 'confidence', 'reasoning', 'description', 'timestamp']

Severity breakdown:
severity
Medium      16
Critical    10
Low          7
High         4
Name: count, dtype: int64

Component breakdown:
component
Payment Processing                    4
UI/Rendering                          4
Cache Management                      4
Initialization                        3
Database                              3
Unknown                               3
Startup/Initialization                2
UI/Preferences                        2
Login                                 2
Search/Filtering                      1
Export/PDF Rendering                  1
Email/Notification System             1
Authentication                        1
Server-Side Logic                     1
Code Compilation/Map Functionality    1
Database/Typescript Configuration     1
Compiler/Linker          

In [5]:
COMPONENT_MERGE_MAP = {
    "Startup/Initialization": "Initialization",
}

df['component_normalized'] = df['component'].replace(COMPONENT_MERGE_MAP)

In [6]:
print(df['component_normalized'].value_counts())

component_normalized
Initialization                        5
Payment Processing                    4
UI/Rendering                          4
Cache Management                      4
Database                              3
Unknown                               3
UI/Preferences                        2
Login                                 2
Search/Filtering                      1
Export/PDF Rendering                  1
Email/Notification System             1
Authentication                        1
Server-Side Logic                     1
Code Compilation/Map Functionality    1
Database/Typescript Configuration     1
Compiler/Linker                       1
Compiler/Borrow Checker               1
API/Backend                           1
Name: count, dtype: int64


## designing of grouping

In [ ]:
# {
#     "total_submissions": 37,
#     "classified_count": 34,
#     "unknown_count": 3,
#     "unknown_rate": 8.1,          # % — the reliability metric

#     "severity_breakdown": [
#         {"label": "Medium", "count": 16, "percent": 43.2},
#         {"label": "Critical", "count": 10, "percent": 27.0},
#         ...
#     ],

#     "component_breakdown": [
#         {"label": "Payment Processing", "count": 4, "percent": 11.8},
#         ...
#     ],

#     "root_cause_breakdown": [
#         {"label": "NullPointerException-family", "count": 5, "percent": 14.7, "bug_ids": ["BUG-...", ...]},
#         ...
#     ]
# }

In [9]:
# Updated final structure (pending your severity check):

# python
# {
#     "total_submissions": 37,
#     "classified_count": 34,
#     "unknown_count": 3,
#     "unknown_rate": 8.1,

#     "severity_breakdown": [...],      # % over 37
#     "component_breakdown": [...],     # % over 34
#     "root_cause_breakdown": [
#         {"label": "...", "count": 5, "percent": 14.7, "bug_ids": [...]}
#     ],

#     "submission_activity": [
#         {"date": "2026-07-15", "count": 32},
#         ...
#     ]
# }

## Grouping

In [ ]:
# Structure, final:
# {
#     "total_submissions": 37,
#     "classified_count": 34,
#     "unknown_count": 3,
#     "unknown_rate": 8.1,

#     "severity_breakdown": [
#         {"label": "Medium", "count": 16, "percent": 43.2},
#         {"label": "Critical", "count": 10, "percent": 27.0},
#         {"label": "Low", "count": 7, "percent": 18.9},
#         {"label": "High", "count": 4, "percent": 10.8},
#     ],

#     "component_breakdown": [
#         {"label": "Payment Processing", "count": 4, "percent": 11.8},
#         ...
#     ],

#     "root_cause_breakdown": [
#         {"label": "NullPointerException-family", "count": 5, "percent": 14.7, "bug_ids": ["BUG-...", ...]},
#         ...
#     ],

#     "submission_activity": [
#         {"date": "2026-07-15", "count": 32},
#         ...
#     ]
# }

In [7]:
print(df['severity'].unique())

<ArrowStringArray>
['Critical', 'High', 'Medium', 'Low']
Length: 4, dtype: str


In [10]:
def compute_defect_analytics(df):
    total = len(df)

    # --- Unknown handling: classified vs unknown, based on component ---
    unknown_mask = df['component'] == 'Unknown'
    unknown_count = int(unknown_mask.sum())
    classified_count = total - unknown_count
    unknown_rate = round((unknown_count / total) * 100, 1) if total > 0 else 0.0

    # --- Severity breakdown: % over ALL submissions (no Unknown contamination here) ---
    severity_counts = df['severity'].value_counts()
    severity_breakdown = [
        {
            "label": label,
            "count": int(count),
            "percent": round((count / total) * 100, 1)
        }
        for label, count in severity_counts.items()
    ]

    # --- Component breakdown: % over CLASSIFIED only, using component_normalized ---
    classified_df = df[~unknown_mask]
    component_counts = classified_df['component_normalized'].value_counts()
    component_breakdown = [
        {
            "label": label,
            "count": int(count),
            "percent": round((count / classified_count) * 100, 1) if classified_count > 0 else 0.0
        }
        for label, count in component_counts.items()
    ]

    # --- Submission activity: count per calendar date (from timestamp) ---
    df_dates = pd.to_datetime(df['timestamp']).dt.date
    activity_counts = df_dates.value_counts().sort_index()
    submission_activity = [
        {"date": str(date), "count": int(count)}
        for date, count in activity_counts.items()
    ]

    return {
        "total_submissions": total,
        "classified_count": classified_count,
        "unknown_count": unknown_count,
        "unknown_rate": unknown_rate,
        "severity_breakdown": severity_breakdown,
        "component_breakdown": component_breakdown,
        "submission_activity": submission_activity,
        # root_cause_breakdown added in step 2
    }

In [11]:
result = compute_defect_analytics(df)
import json
print(json.dumps(result, indent=2))

{
  "total_submissions": 37,
  "classified_count": 34,
  "unknown_count": 3,
  "unknown_rate": 8.1,
  "severity_breakdown": [
    {
      "label": "Medium",
      "count": 16,
      "percent": 43.2
    },
    {
      "label": "Critical",
      "count": 10,
      "percent": 27.0
    },
    {
      "label": "Low",
      "count": 7,
      "percent": 18.9
    },
    {
      "label": "High",
      "count": 4,
      "percent": 10.8
    }
  ],
  "component_breakdown": [
    {
      "label": "Initialization",
      "count": 5,
      "percent": 14.7
    },
    {
      "label": "Payment Processing",
      "count": 4,
      "percent": 11.8
    },
    {
      "label": "UI/Rendering",
      "count": 4,
      "percent": 11.8
    },
    {
      "label": "Cache Management",
      "count": 4,
      "percent": 11.8
    },
    {
      "label": "Database",
      "count": 3,
      "percent": 8.8
    },
    {
      "label": "UI/Preferences",
      "count": 2,
      "percent": 5.9
    },
    {
      "label":

In [13]:
import sqlite3

conn = sqlite3.connect("../data/bug_submissions.db")
cursor = conn.cursor()
cursor.execute("ALTER TABLE bug_submissions ADD COLUMN root_cause_hypothesis TEXT")
conn.commit()
conn.close()

print("Column added.")

Column added.


In [14]:
conn = sqlite3.connect("../data/bug_submissions.db")
df_check = pd.read_sql("SELECT * FROM bug_submissions", conn)
conn.close()
print(df_check.columns.tolist())

['bug_id', 'severity', 'priority', 'component', 'error_type', 'failure_location', 'code_path', 'confidence', 'reasoning', 'description', 'timestamp', 'root_cause_hypothesis']


In [15]:
conn = sqlite3.connect("../data/bug_submissions.db")
df_check = pd.read_sql("SELECT bug_id, component, root_cause_hypothesis FROM bug_submissions WHERE bug_id IN ('BUG-6d4769ce', 'BUG-84f36110')", conn)
conn.close()
print(df_check)

         bug_id    component  \
0  BUG-6d4769ce   PDF Export   
1  BUG-84f36110  File Upload   

                               root_cause_hypothesis  
0  The PDF Export Formatting Error is likely caus...  
1  The MemoryError likely occurred because the bu...  


In [16]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

reasoning_texts = df['reasoning'].fillna("").tolist()
reasoning_embeddings = model.encode(reasoning_texts, convert_to_numpy=True)

sim_matrix = cosine_similarity(reasoning_embeddings)

# Get only the upper triangle, excluding the diagonal (a bug compared to itself = always 1.0, not useful)
upper_triangle = sim_matrix[np.triu_indices(len(sim_matrix), k=1)]

print("Number of unique pairs:", len(upper_triangle))
print("Min similarity:", upper_triangle.min())
print("Max similarity:", upper_triangle.max())
print("Mean similarity:", upper_triangle.mean())
print("Median similarity:", np.median(upper_triangle))
print("\nPercentiles:")
for p in [50, 70, 80, 90, 95, 99]:
    print(f"{p}th percentile: {np.percentile(upper_triangle, p):.3f}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Number of unique pairs: 666
Min similarity: -0.120762646
Max similarity: 1.0000004
Mean similarity: 0.19506845
Median similarity: 0.15426737

Percentiles:
50th percentile: 0.154
70th percentile: 0.209
80th percentile: 0.261
90th percentile: 0.357
95th percentile: 0.706
99th percentile: 1.000


In [17]:
from sklearn.cluster import AgglomerativeClustering

clustering = AgglomerativeClustering(
    n_clusters=None,
    distance_threshold=0.35,   # 1 - 0.65 similarity
    metric='cosine',
    linkage='average'
)
labels = clustering.fit_predict(reasoning_embeddings)

df['cluster_id'] = labels
print("Number of clusters formed:", len(set(labels)))
print(df['cluster_id'].value_counts())

Number of clusters formed: 19
cluster_id
1     5
7     4
2     4
6     4
3     3
5     3
0     2
14    1
15    1
11    1
12    1
9     1
13    1
16    1
17    1
10    1
18    1
8     1
4     1
Name: count, dtype: int64


In [18]:
print(df[df['cluster_id'] == 1][['bug_id', 'component', 'error_type', 'reasoning']].to_string())

          bug_id               component            error_type                                                                                                                                                                                                                   reasoning
0   BUG-3b0fe790          Initialization  NullPointerException  The program attempts to access or manipulate a null object reference during the initialization process in the Init class, resulting in a NullPointerException that causes the application to crash immediately on startup.
5   BUG-1f12378b          Initialization  NullPointerException              The application crashes on startup due to a NullPointerException, indicating that a null object reference is being dereferenced at line 22 of Init.java, likely caused by an uninitialized variable or object.
9   BUG-c53ac751  Startup/Initialization  NullPointerException                                   The program attempted to access or manipulate a null (

In [19]:
print(df[df['cluster_id'] == 0][['bug_id', 'component', 'error_type', 'reasoning']].to_string())

          bug_id       component            error_type                                                                                                                                           reasoning
18  BUG-1a9af3a8  UI/Preferences  Not an Error Message                     The input is a feature request, not an error message, and does not contain any information about a specific error or exception.
20  BUG-79107ce5  UI/Preferences  Not an Error Message  The input is a feature request and does not contain any error or exception information, so no error type, location, or code path can be extracted.


In [20]:
#outdated one 

def cluster_root_causes(df, distance_threshold=0.35):
    reasoning_texts = df['reasoning'].fillna("").tolist()

    model = SentenceTransformer('all-MiniLM-L6-v2')
    reasoning_embeddings = model.encode(reasoning_texts, convert_to_numpy=True)

    clustering = AgglomerativeClustering(
        n_clusters=None,
        distance_threshold=distance_threshold,
        metric='cosine',
        linkage='average'
    )
    cluster_labels = clustering.fit_predict(reasoning_embeddings)

    df_temp = df.copy()
    df_temp['cluster_id'] = cluster_labels

    total = len(df_temp)
    root_cause_breakdown = []

    for cluster_id in sorted(df_temp['cluster_id'].unique()):
        cluster_rows = df_temp[df_temp['cluster_id'] == cluster_id]
        count = len(cluster_rows)

        # Label = most frequent error_type within this cluster
        label = cluster_rows['error_type'].value_counts().idxmax()

        root_cause_breakdown.append({
            "label": label,
            "count": int(count),
            "percent": round((count / total) * 100, 1),
            "bug_ids": cluster_rows['bug_id'].tolist()
        })

    # Sort by count descending, biggest clusters first
    root_cause_breakdown.sort(key=lambda x: x["count"], reverse=True)

    return root_cause_breakdown

In [ ]:
# update one 

def cluster_root_causes(df, distance_threshold=0.35):
    reasoning_texts = df['reasoning'].fillna("").tolist()

    model = SentenceTransformer('all-MiniLM-L6-v2')
    reasoning_embeddings = model.encode(reasoning_texts, convert_to_numpy=True)

    clustering = AgglomerativeClustering(
        n_clusters=None,
        distance_threshold=distance_threshold,
        metric='cosine',
        linkage='average'
    )
    cluster_labels = clustering.fit_predict(reasoning_embeddings)

    df_temp = df.copy()
    df_temp['cluster_id'] = cluster_labels

    total = len(df_temp)
    root_cause_breakdown = []

    for cluster_id in sorted(df_temp['cluster_id'].unique()):
        cluster_rows = df_temp[df_temp['cluster_id'] == cluster_id]
        count = len(cluster_rows)

        error_type_counts = cluster_rows['error_type'].value_counts()
        top_label = error_type_counts.idxmax()
        distinct_error_types = error_type_counts.index.tolist()

        # Fix: rename the error_type "Unknown" (Log Analysis extraction failure)
        # so it's never confused with component-level "Unknown" (Triage failure)
        if top_label == "Unknown":
            top_label = "Unclassified (Log Analysis extraction failed)"

        root_cause_breakdown.append({
            "label": top_label,
            "count": int(count),
            "percent": round((count / total) * 100, 1),
            "bug_ids": cluster_rows['bug_id'].tolist(),
            "distinct_error_types": distinct_error_types
        })

    root_cause_breakdown.sort(key=lambda x: x["count"], reverse=True)

    return root_cause_breakdown

In [24]:
root_causes = cluster_root_causes(df)
import json
print(json.dumps(root_causes, indent=2))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[
  {
    "label": "NullPointerException",
    "count": 5,
    "percent": 13.5,
    "bug_ids": [
      "BUG-3b0fe790",
      "BUG-1f12378b",
      "BUG-c53ac751",
      "BUG-09e0269c",
      "BUG-a5336770"
    ],
    "distinct_error_types": [
      "NullPointerException"
    ]
  },
  {
    "label": "NameError",
    "count": 4,
    "percent": 10.8,
    "bug_ids": [
      "BUG-4f3ac660",
      "BUG-c1a805a2",
      "BUG-da34fe7f",
      "BUG-daad1134"
    ],
    "distinct_error_types": [
      "NameError"
    ]
  },
  {
    "label": "Unclassified (Log Analysis extraction failed)",
    "count": 4,
    "percent": 10.8,
    "bug_ids": [
      "BUG-fae83d0c",
      "BUG-176f4fed",
      "BUG-ef822c82",
      "BUG-42596b47"
    ],
    "distinct_error_types": [
      "Unknown"
    ]
  },
  {
    "label": "ValidationError",
    "count": 4,
    "percent": 10.8,
    "bug_ids": [
      "BUG-b216f3db",
      "BUG-f5b5592a",
      "BUG-f6c01d15",
      "BUG-b48d66f1"
    ],
    "distinct_error_types

In [22]:
print(df[df['bug_id'].isin(['BUG-173ed07b', 'BUG-cf3508db', 'BUG-8956127d'])][['bug_id', 'error_type', 'reasoning']].to_string())

          bug_id           error_type                                                                                                                                                              reasoning
4   BUG-173ed07b        Cosmetic Typo  A minor typo exists in the button label text, displaying "Sumbit" instead of the correct spelling "Submit", which is a cosmetic issue rather than a functional error.
10  BUG-cf3508db  Typographical Error                     A typo in the button label text, "Sumbit" instead of "Submit", is a cosmetic issue that does not affect functionality but impacts user experience.
21  BUG-8956127d  Typographical Error                     A typo in the button label text, "Sumbit" instead of "Submit", is a cosmetic issue that does not affect functionality but impacts user experience.


In [25]:
def compute_defect_analytics(df, distance_threshold=0.35):
    total = len(df)

    # --- Unknown handling: classified vs unknown, based on component ---
    unknown_mask = df['component'] == 'Unknown'
    unknown_count = int(unknown_mask.sum())
    classified_count = total - unknown_count
    unknown_rate = round((unknown_count / total) * 100, 1) if total > 0 else 0.0

    # --- Severity breakdown: % over ALL submissions ---
    severity_counts = df['severity'].value_counts()
    severity_breakdown = [
        {
            "label": label,
            "count": int(count),
            "percent": round((count / total) * 100, 1)
        }
        for label, count in severity_counts.items()
    ]

    # --- Component breakdown: % over CLASSIFIED only, using component_normalized ---
    classified_df = df[~unknown_mask]
    component_counts = classified_df['component_normalized'].value_counts()
    component_breakdown = [
        {
            "label": label,
            "count": int(count),
            "percent": round((count / classified_count) * 100, 1) if classified_count > 0 else 0.0
        }
        for label, count in component_counts.items()
    ]

    # --- Submission activity: count per calendar date ---
    df_dates = pd.to_datetime(df['timestamp']).dt.date
    activity_counts = df_dates.value_counts().sort_index()
    submission_activity = [
        {"date": str(date), "count": int(count)}
        for date, count in activity_counts.items()
    ]

    # --- Root cause breakdown: semantic clustering on 'reasoning' ---
    root_cause_breakdown = cluster_root_causes(df, distance_threshold=distance_threshold)

    return {
        "total_submissions": total,
        "classified_count": classified_count,
        "unknown_count": unknown_count,
        "unknown_rate": unknown_rate,
        "severity_breakdown": severity_breakdown,
        "component_breakdown": component_breakdown,
        "root_cause_breakdown": root_cause_breakdown,
        "submission_activity": submission_activity
    }

In [30]:
result = compute_defect_analytics(df)
import json
print(json.dumps(result, indent=2))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

{
  "total_submissions": 39,
  "classified_count": 36,
  "unknown_count": 3,
  "unknown_rate": 7.7,
  "severity_breakdown": [
    {
      "label": "Medium",
      "count": 16,
      "percent": 41.0
    },
    {
      "label": "Critical",
      "count": 10,
      "percent": 25.6
    },
    {
      "label": "Low",
      "count": 8,
      "percent": 20.5
    },
    {
      "label": "High",
      "count": 5,
      "percent": 12.8
    }
  ],
  "component_breakdown": [
    {
      "label": "Initialization",
      "count": 5,
      "percent": 13.9
    },
    {
      "label": "Payment Processing",
      "count": 4,
      "percent": 11.1
    },
    {
      "label": "UI/Rendering",
      "count": 4,
      "percent": 11.1
    },
    {
      "label": "Cache Management",
      "count": 4,
      "percent": 11.1
    },
    {
      "label": "Database",
      "count": 3,
      "percent": 8.3
    },
    {
      "label": "UI/Preferences",
      "count": 2,
      "percent": 5.6
    },
    {
      "label":

## Task 2 : UI 

In [1]:
import sqlite3

conn = sqlite3.connect("../data/bug_submissions.db")
cursor = conn.cursor()
cursor.execute("ALTER TABLE bug_submissions ADD COLUMN status TEXT DEFAULT 'pending_review'")
cursor.execute("ALTER TABLE bug_submissions ADD COLUMN recommended_fix TEXT")
conn.commit()
conn.close()

print("Columns added.")

Columns added.


In [3]:
import pandas as pd
conn = sqlite3.connect("../data/bug_submissions.db")
df_check = pd.read_sql("SELECT * FROM bug_submissions", conn)
conn.close()
print(df_check.columns.tolist())
print(df_check['status'].value_counts())

['bug_id', 'severity', 'priority', 'component', 'error_type', 'failure_location', 'code_path', 'confidence', 'reasoning', 'description', 'timestamp', 'root_cause_hypothesis', 'status', 'recommended_fix']
status
pending_review    42
Name: count, dtype: int64


## Loading and updatin the sqlite DB

In [27]:
conn = sqlite3.connect("../data/bug_submissions.db")
df = pd.read_sql("SELECT * FROM bug_submissions", conn)
conn.close()
print("Total bugs in database:", len(df))

Total bugs in database: 39


In [28]:
df['component_normalized'] = df['component'].replace(COMPONENT_MERGE_MAP)

In [29]:
result = compute_defect_analytics(df)
print(json.dumps(result, indent=2))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

{
  "total_submissions": 39,
  "classified_count": 36,
  "unknown_count": 3,
  "unknown_rate": 7.7,
  "severity_breakdown": [
    {
      "label": "Medium",
      "count": 16,
      "percent": 41.0
    },
    {
      "label": "Critical",
      "count": 10,
      "percent": 25.6
    },
    {
      "label": "Low",
      "count": 8,
      "percent": 20.5
    },
    {
      "label": "High",
      "count": 5,
      "percent": 12.8
    }
  ],
  "component_breakdown": [
    {
      "label": "Initialization",
      "count": 5,
      "percent": 13.9
    },
    {
      "label": "Payment Processing",
      "count": 4,
      "percent": 11.1
    },
    {
      "label": "UI/Rendering",
      "count": 4,
      "percent": 11.1
    },
    {
      "label": "Cache Management",
      "count": 4,
      "percent": 11.1
    },
    {
      "label": "Database",
      "count": 3,
      "percent": 8.3
    },
    {
      "label": "UI/Preferences",
      "count": 2,
      "percent": 5.6
    },
    {
      "label":

In [6]:
conn = sqlite3.connect("../data/bug_submissions.db")
df_check = pd.read_sql("SELECT bug_id, status, recommended_fix FROM bug_submissions ORDER BY timestamp DESC LIMIT 1", conn)
conn.close()
print(df_check)

         bug_id          status  \
0  BUG-f80403a9  pending_review   

                                     recommended_fix  
0  Check if the path is a file before attempting ...  


In [7]:
import pandas as pd
metadata = pd.read_csv("../src/chunks_metadata.csv")
print(metadata.columns.tolist())
print(metadata.head(2))

['bug_id', 'title', 'severity', 'resolution', 'status', 'source_dataset']
                      bug_id  \
0            BUGZILLA-294734   
1  OTHER_APPLICATIONS-363323   

                                               title severity resolution  \
0                          Emergency 2.16.10 Release  blocker      fixed   
1  DOM View is really inefficient with setting wh...   normal      fixed   

     status source_dataset  
0  resolved        Mozilla  
1  resolved        Mozilla  


# Check  , validation , and testing

## re run codes

In [2]:
print(sorted(df['component'].unique()))

['API/Backend', 'Authentication', 'Cache Management', 'Code Compilation/Map Functionality', 'Compiler/Borrow Checker', 'Compiler/Linker', 'Database', 'Database/Typescript Configuration', 'Email/Notification System', 'Export/PDF Rendering', 'Initialization', 'Login', 'Payment Processing', 'Search/Filtering', 'Server-Side Logic', 'Startup/Initialization', 'UI/Preferences', 'UI/Rendering', 'Unknown']


In [4]:
print(df[df['component'] == 'Database/Typescript Configuration']['description'].values)


<ArrowStringArray>
['"types_4.ts:2:5 - error TS2564: Property 'connectionString' has no initializer and is not definitely assigned in the constructor.\n\n2     connectionString: string;\n      ~~~~~~~~~~~~~~~~\n"']
Length: 1, dtype: str


In [12]:
print(df.columns.tolist())

['bug_id', 'severity', 'priority', 'component', 'error_type', 'failure_location', 'code_path', 'confidence', 'reasoning', 'description', 'timestamp', 'component_normalized']


In [31]:
conn = sqlite3.connect("../data/bug_submissions.db")
df_check = pd.read_sql("SELECT * FROM bug_submissions", conn)
conn.close()

print(df_check[df_check['error_type'].str.contains(',', na=False)][['bug_id', 'description', 'error_type', 'reasoning']].to_string())

          bug_id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        description                                                           error_type                                                                                                                                                                                

In [13]:
metadata = pd.read_csv("../src/chunks_metadata.csv")
print(len(metadata))
print(metadata.tail(2))

56864
             bug_id      title severity  \
56862  BUG-d5b55cac  NameError   Medium   
56863  BUG-a6564d59  NameError   Medium   

                                              resolution    status  \
56862  Define the function 'adddd' before it is calle...  resolved   
56863  Correct the typo in the function name from 'su...  resolved   

         source_dataset  
56862  live_submissions  
56863  live_submissions  


In [1]:
import os
import pandas as pd

RANDOM_SEED = 42
UPLOAD_DIR1 = "path"
UPLOAD_DIR2 = "path"
UPLOAD_DIR3 = "path"



LANGUAGE_FILES = {""
    "Python": "python_10_out_of_10_dataset_500.csv",
    "Java": "java_smart_analyzer_dataset_500.csv",
    "Rust": "rust_smart_analyzer_dataset_v3_500.csv",
    "Go": "go_smart_analyzer_dataset_500.csv",
    "TypeScript": "ts_smart_analyzer_dataset_500.csv",
    "PHP": "php_smart_analyzer_dataset_500.csv",
    "CSS": "css_smart_analyzer_dataset_v2_500.csv",
    "HTML": "html_smart_analyzer_dataset_v2_500.csv",
    "C++": "cpp_smart_analyzer_dataset_500.csv",
    "C": "c_smart_analyzer_dataset_500.csv",
    "JavaScript": "js_smart_analyzer_dataset_500.csv",
}

test_bugs = []

# --- 11 language files: 3 random rows each ---
for lang, filename in LANGUAGE_FILES.items():
    df = pd.read_csv(f"{UPLOAD_DIR1}/{filename}")
    sample = df.sample(n=3, random_state=RANDOM_SEED)
    for _, row in sample.iterrows():
        combined_text = f"{row['code']}\n\n{row['stack_trace']}"
        test_bugs.append({
            "source": lang,
            "text": combined_text,
            "expected_error_type": row["error_type"],
            "expected_fix_brief": row["fix_brief"]
        })

# --- Mozilla: 2 random rows ---
mozilla_df = pd.read_csv(f"{UPLOAD_DIR2}/mozilla_bug_report_data.csv")
mozilla_sample = mozilla_df.sample(n=2, random_state=RANDOM_SEED)
for _, row in mozilla_sample.iterrows():
    combined_text = f"{row['short_description']}\n\n{row['long_description']}"
    test_bugs.append({
        "source": "Mozilla",
        "text": combined_text,
        "expected_error_type": row.get("severity_category", "N/A"),
        "expected_fix_brief": row.get("resolution_category", "N/A")
    })

# --- Eclipse: 2 random rows ---
eclipse_df = pd.read_csv(f"{UPLOAD_DIR3}/eclipse_bug_report_data.csv")
eclipse_sample = eclipse_df.sample(n=2, random_state=RANDOM_SEED)
for _, row in eclipse_sample.iterrows():
    combined_text = f"{row['short_description']}\n\n{row['long_description']}"
    test_bugs.append({
        "source": "Eclipse",
        "text": combined_text,
        "expected_error_type": row.get("severity_category", "N/A"),
        "expected_fix_brief": row.get("resolution_category", "N/A")
    })

# --- Apache (processed_issues.csv): 2 random rows ---
apache_df = pd.read_csv(f"{UPLOAD_DIR1}/apache_processed_issues.csv")
apache_sample = apache_df.sample(n=2, random_state=RANDOM_SEED)
for _, row in apache_sample.iterrows():
    combined_text = f"{row['summary']}\n\n{row['description']}"
    test_bugs.append({
        "source": f"Apache ({row.get('project.name', 'unknown')})",
        "text": combined_text,
        "expected_error_type": "N/A",
        "expected_fix_brief": row.get("resolution.name", "N/A")
    })

print(f"Total test bugs sampled: {len(test_bugs)}")
for i, bug in enumerate(test_bugs, 1):
    print(f"{i}. [{bug['source']}] {bug['text'][:80].strip()!r}...")


Total test bugs sampled: 39
1. [Python] "def process_records_362():\n    with open('/etc/shadow', 'w') as file:\n        fi"...
2. [Python] "def convert_encoding_74():\n    session_token_74 = 'test_string'\n    return sessi"...
3. [Python] "def fetch_data_375(val_str):\n    return float(val_str)\n\nfetch_data_375('invalid_"...
4. [Java] 'public class SessionManager_362 {\n    public void authenticateUser_362() {'...
5. [Java] 'public class ResourceGuard_74 {\n    public static void main(String[] args) {'...
6. [Java] 'public class DatabaseConnector_375 {\n    public static void main(String[] args)'...
7. [Rust] 'fn main() {\n    let mut records_362 = vec![362, 363, 364];\n    let r_362 = &reco'...
8. [Rust] 'fn compute_val_74(x: i32) -> i32 {\n    x + 74;\n}\n\nerror[E0308]: mismatched types'...
9. [Rust] 'fn main() {\n    let profile_375 = vec![10, 20, 30];\n    println!("{}", profile_3'...
10. [Go] 'package main\nimport "fmt"\nfunc main() {\n    sensorReading_362 := []int{1, 2, 3}'..

In [3]:
import pandas as pd
kb_metadata_check = pd.read_csv("../src/chunks_metadata.csv")
print(kb_metadata_check.columns.tolist())

['bug_id', 'title', 'severity', 'resolution', 'status', 'source_dataset']


In [2]:
# # import sys
# # import os

# # # Option A: Using the absolute (full) path to folder_c
# # sys.path.append(r"rootpath/AI_Smart_Bug_Analyzer_-_Fix-_Advisor-/src")
# # import importlib
# # import embeddings_real
# # importlib.reload(embeddings_real)

# # import numpy as np

# # # Load the numpy array file
# # embeddings = np.load('embeddings_real.npy')

# # # View the contents and its structure
# # print(type(embeddings))
# # print(embeddings.shape)
# # print(embeddings)


import os
import sys
import numpy as np

 # 1. Provide the actual absolute path starting with your drive letter
src_path = r"path"
sys.path.append(src_path)

# # 2. Combine the path directly into np.load so it knows exactly where the file lives
# file_path = os.path.join(src_path, "embeddings_real.npy")

# # 3. Load and inspect the file safely
# if os.path.exists(file_path):
#     kb_embeddings = np.load(file_path)
#     print(f"Data type: {type(kb_embeddings)}")
#     print(f"Array shape: {kb_embeddings.shape}")
#     print(kb_embeddings)
# else:
#     print(f"Error: Could not find the file at {file_path}")



In [3]:
from sentence_transformers import SentenceTransformer
import numpy as np
import pandas as pd

model = SentenceTransformer('all-MiniLM-L6-v2')
kb_embeddings = np.load("../src/embeddings_real.npy")
kb_metadata = pd.read_csv("../src/chunks_metadata.csv")

print("Model loaded")
print("Embeddings shape:", kb_embeddings.shape)
print("Metadata shape:", kb_metadata.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded
Embeddings shape: (56864, 384)
Metadata shape: (56864, 6)


In [77]:
# old

import time

from agents import (
    triage_agent,log_analysis_agent, build_simple_view, init_db, save_submission,
    retrieve_similar_bugs, root_cause_agent,
    duplicate_detection_agent, remediation_agent,
    COMPONENT_MERGE_MAP, compute_defect_analytics, cluster_root_causes,
    add_resolved_bug_to_kb
)


# Make sure these are already imported/available from your agents.py, 
# and that `model` (SentenceTransformer) is already loaded in this notebook,
# same as your earlier cells.

test_results = []

def run_one_test(bug, index, top_n=5):
    bug_id = f"TEST-{index:02d}"
    text = bug["text"]

    print(f"\n{'='*80}")
    print(f"TEST #{index} — Source: {bug['source']}")
    print(f"{'='*80}")

    combined_result = run_orchestration(
        title=text[:80], description=text, stack_trace=text, bug_id=bug_id
    )
    triage = combined_result["triage"]
    log = combined_result["log_analysis"]

    print(f"Severity: {triage['severity']} | Component: {triage['component']}")
    print(f"Error Type: {log['error_type']} | Location: {log['failure_location']}")

    retrieved = retrieve_similar_bugs(text, model, kb_embeddings, kb_metadata, top_n=top_n)

    root_cause = root_cause_agent(
        bug_id=bug_id, severity=triage["severity"], component=triage["component"],
        error_type=log["error_type"], failure_location=log["failure_location"],
        code_path=log["code_path"], retrieved_bugs=retrieved
    )
    print(f"\nRoot Cause: {root_cause['root_cause_hypothesis'][:200]}")
    print(f"Confidence: {root_cause['confidence']}")

    remediation = remediation_agent(
        bug_id=bug_id, severity=triage["severity"], component=triage["component"],
        error_type=log["error_type"], failure_location=log["failure_location"],
        code_path=log["code_path"], description=text,
        root_cause=root_cause["root_cause_hypothesis"],
        historical_references=root_cause["supporting_evidence"],
        duplicate_bug=None
    )
    print(f"\nRecommended Fix: {remediation['recommended_fix']}")
    print(f"References used: {remediation.get('references_used', [])}")
    print(f"\n--- Expected (from dataset) ---")
    print(f"Expected error type: {bug['expected_error_type']}")
    print(f"Expected fix: {bug['expected_fix_brief']}")

    return {
        "index": index, "source": bug["source"], "bug_id": bug_id,
        "severity": triage["severity"], "error_type": log["error_type"],
        "root_cause_confidence": root_cause["confidence"],
        "recommended_fix": remediation["recommended_fix"],
        "references_used": remediation.get("references_used", [])
    }

In [73]:

#new one updated 1

import time

def run_one_test(bug, index, top_n=5):
    bug_id = f"TEST-{index:02d}"
    text = bug["text"]

    print(f"\n{'='*80}")
    print(f"TEST #{index} — Source: {bug['source']}")
    print(f"{'='*80}")

    combined_result = run_orchestration(
        title=text[:80], description=text, stack_trace=text, bug_id=bug_id
    )
    time.sleep(3)   # <-- new pause after Triage+LogAnalysis

    triage = combined_result["triage"]
    log = combined_result["log_analysis"]

    print(f"Severity: {triage['severity']} | Component: {triage['component']}")
    print(f"Error Type: {log['error_type']} | Location: {log['failure_location']}")

    retrieved = retrieve_similar_bugs(text, model, kb_embeddings, kb_metadata, top_n=top_n)

    root_cause = root_cause_agent(
        bug_id=bug_id, severity=triage["severity"], component=triage["component"],
        error_type=log["error_type"], failure_location=log["failure_location"],
        code_path=log["code_path"], retrieved_bugs=retrieved
    )
    time.sleep(3)   # <-- new pause after Root Cause

    print(f"\nRoot Cause: {root_cause['root_cause_hypothesis'][:200]}")
    print(f"Confidence: {root_cause['confidence']}")

    remediation = remediation_agent(
        bug_id=bug_id, severity=triage["severity"], component=triage["component"],
        error_type=log["error_type"], failure_location=log["failure_location"],
        code_path=log["code_path"], description=text,
        root_cause=root_cause["root_cause_hypothesis"],
        historical_references=root_cause["supporting_evidence"],
        duplicate_bug=None
    )
    print(f"\nRecommended Fix: {remediation['recommended_fix']}")
    print(f"References used: {remediation.get('references_used', [])}")
    print(f"\n--- Expected (from dataset) ---")
    print(f"Expected error type: {bug['expected_error_type']}")
    print(f"Expected fix: {bug['expected_fix_brief']}")

    return {
        "index": index, "source": bug["source"], "bug_id": bug_id,
        "severity": triage["severity"], "error_type": log["error_type"],
        "root_cause_confidence": root_cause["confidence"],
        "recommended_fix": remediation["recommended_fix"],
        "references_used": remediation.get("references_used", [])
    }

In [4]:
# updated 2nd time 

import time


import time

from agents import (
    triage_agent,log_analysis_agent, build_simple_view, init_db, save_submission,
    retrieve_similar_bugs, root_cause_agent,
    duplicate_detection_agent, remediation_agent,
    COMPONENT_MERGE_MAP, compute_defect_analytics, cluster_root_causes,
    add_resolved_bug_to_kb
)

def run_one_test(bug, index, top_n=5):
    bug_id = f"TEST-{index:02d}"
    text = bug["text"]

    print(f"\n{'='*80}")
    print(f"TEST #{index} — Source: {bug['source']}")
    print(f"{'='*80}")

    triage = triage_agent(title=text[:80], description=text, stack_trace=text, bug_id=bug_id)
    time.sleep(6)

    log = log_analysis_agent(text)
    time.sleep(6)

    print(f"Severity: {triage['severity']} | Component: {triage['component']}")
    print(f"Error Type: {log['error_type']} | Location: {log['failure_location']}")

    retrieved = retrieve_similar_bugs(text, model, kb_embeddings, kb_metadata, top_n=top_n)

    root_cause = root_cause_agent(
        bug_id=bug_id, severity=triage["severity"], component=triage["component"],
        error_type=log["error_type"], failure_location=log["failure_location"],
        code_path=log["code_path"], retrieved_bugs=retrieved
    )
    time.sleep(6)

    print(f"\nRoot Cause: {root_cause['root_cause_hypothesis'][:200]}")
    print(f"Confidence: {root_cause['confidence']}")

    remediation = remediation_agent(
        bug_id=bug_id, severity=triage["severity"], component=triage["component"],
        error_type=log["error_type"], failure_location=log["failure_location"],
        code_path=log["code_path"], description=text,
        root_cause=root_cause["root_cause_hypothesis"],
        historical_references=root_cause["supporting_evidence"],
        duplicate_bug=None
    )
    print(f"\nRecommended Fix: {remediation['recommended_fix']}")
    print(f"References used: {remediation.get('references_used', [])}")
    print(f"\n--- Expected (from dataset) ---")
    print(f"Expected error type: {bug['expected_error_type']}")
    print(f"Expected fix: {bug['expected_fix_brief']}")

    return {
        "index": index, "source": bug["source"], "bug_id": bug_id,
        "severity": triage["severity"], "error_type": log["error_type"],
        "root_cause_confidence": root_cause["confidence"],
        "recommended_fix": remediation["recommended_fix"],
        "references_used": remediation.get("references_used", [])
    }

In [5]:
from agents import call_llm_with_retry, log_analysis_system_prompt
from agents import client

text = test_bugs[11]["text"]
print("--- INPUT TEXT ---")
print(text)
print()

user_message = f"Error:\n{text}"

try:
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": log_analysis_system_prompt},
            {"role": "user", "content": user_message}
        ],
        temperature=0.2
    )
    raw = response.choices[0].message.content
    print("--- RAW LLM OUTPUT (before any parsing) ---")
    print(raw)
except Exception as e:
    print("--- EXCEPTION ---")
    print(type(e).__name__, ":", str(e))
    
from agents import call_llm_with_retry, log_analysis_system_prompt
from agents import client

text = test_bugs[11]["text"]
print("--- INPUT TEXT ---")
print(text)
print()

user_message = f"Error:\n{text}"

try:
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": log_analysis_system_prompt},
            {"role": "user", "content": user_message}
        ],
        temperature=0.2
    )
    raw = response.choices[0].message.content
    print("--- RAW LLM OUTPUT (before any parsing) ---")
    print(raw)
except Exception as e:
    print("--- EXCEPTION ---")
    print(type(e).__name__, ":", str(e))

--- INPUT TEXT ---
package main
import (
    "fmt"
    "math"
)
func main() {
    fmt.Println("Hello")
}

./handlers_375.go:4:5: "math" imported and not used

--- RAW LLM OUTPUT (before any parsing) ---
{"error_type": "Unused Import", "failure_location": "./handlers_375.go:4:5", "code_path": "main package import -> unused import: \"math\"", "confidence": 0.99, "reasoning": "The 'math' package was imported but none of its functions or variables were used in the code, resulting in an unused import error."}
--- INPUT TEXT ---
package main
import (
    "fmt"
    "math"
)
func main() {
    fmt.Println("Hello")
}

./handlers_375.go:4:5: "math" imported and not used

--- RAW LLM OUTPUT (before any parsing) ---
{"error_type": "Unused Import", "failure_location": "./handlers_375.go:4:5", "code_path": "main package import -> unused import: \"math\"", "confidence": 0.99, "reasoning": "The 'math' package was imported but none of its functions or variables were used in the code, resulting in an unu

In [11]:
from agents import call_llm_with_retry, log_analysis_system_prompt
from agents import client

text = test_bugs[12]["text"]
print("--- INPUT TEXT ---")
print(text)
print()

user_message = f"Error:\n{text}"

try:
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": log_analysis_system_prompt},
            {"role": "user", "content": user_message}
        ],
        temperature=0.2
    )
    raw = response.choices[0].message.content
    print("--- RAW LLM OUTPUT (before any parsing) ---")
    print(raw)
except Exception as e:
    print("--- EXCEPTION ---")
    print(type(e).__name__, ":", str(e))

--- INPUT TEXT ---
const dbCursor_362: object = { id: 362, value: "data_362" };
console.log(dbCursor_362.value);

utils_362.ts:34:13 - error TS2339: Property 'value' does not exist on type 'object'.

34 console.log(dbCursor_362.value);
                         ~~~~~


--- RAW LLM OUTPUT (before any parsing) ---
{"error_type": "TypeScript Error: TS2339 (Property Does Not Exist)", "failure_location": "utils_362.ts:34:13", "code_path": "Top-level execution -> console.log(dbCursor_362.value)", "confidence": 0.99, "reasoning": "The TypeScript compiler is unable to find a 'value' property on the 'dbCursor_362' object because it is declared with the generic type 'object', which does not include any specific properties."}


In [83]:
from agents import call_llm_with_retry, log_analysis_system_prompt
from agents import client

text = test_bugs[13]["text"]
print("--- INPUT TEXT ---")
print(text)
print()

user_message = f"Error:\n{text}"

try:
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": log_analysis_system_prompt},
            {"role": "user", "content": user_message}
        ],
        temperature=0.2
    )
    raw = response.choices[0].message.content
    print("--- RAW LLM OUTPUT (before any parsing) ---")
    print(raw)
except Exception as e:
    print("--- EXCEPTION ---")
    print(type(e).__name__, ":", str(e))

--- INPUT TEXT ---
class Database {
    connectionString: string;
    constructor() {}
}

types_74.ts:2:5 - error TS2564: Property 'connectionString' has no initializer and is not definitely assigned in the constructor.

2     connectionString: string;
      ~~~~~~~~~~~~~~~~


--- RAW LLM OUTPUT (before any parsing) ---
{"error_type": "TypeScript Error: TS2564 (Property Not Definitely Assigned)", "failure_location": "types_74.ts:2:5", "code_path": "class Database -> property declaration: connectionString: string", "confidence": 0.99, "reasoning": "The 'connectionString' property is declared but not initialized in the constructor, and TypeScript cannot guarantee it will be assigned a value before it is used, violating the 'definite assignment' rule."}


In [ ]:
import os
print("Current working directory:", os.getcwd())
print("Looking for .env at ../.env — exists:", os.path.exists("../.env"))

In [33]:
env_content = "GROQ_API_KEY=API_Key_paste_here"

with open("../.env", "w") as f:
    f.write(env_content)

print("Rewrote .env file")

Rewrote .env file


In [ ]:
from dotenv import load_dotenv
load_dotenv("../.env", override=True)
api_key = os.getenv("GROQ_API_KEY")
print("API key loaded:", api_key[:8] if api_key else "NOT FOUND")

In [35]:
from groq import Groq
import json
client = Groq(api_key=api_key)
print("Client created successfully")

Client created successfully


In [36]:
def call_llm_with_retry(system_prompt, user_message, max_retries=3):
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_message}
                ],
                temperature=0.2
            )
            return json.loads(response.choices[0].message.content)
        except Exception as e:
            if "rate_limit" in str(e).lower() or "429" in str(e):
                wait_time = (attempt + 1) * 3
                print(f"Rate limited, waiting {wait_time}s...")
                time.sleep(wait_time)
            else:
                print(f"Non-rate-limit error: {e}")
                break
    return None

In [38]:
model="llama-3.3-70b-versatile"

In [ ]:
# --- Imports ---
import os
import json
import time
import sqlite3
import pandas as pd
import numpy as np
from groq import Groq
from dotenv import load_dotenv
from sklearn.metrics.pairwise import cosine_similarity

# --- API client setup ---
load_dotenv("../.env", override=True)
api_key = os.getenv("GROQ_API_KEY")
client = Groq(api_key=api_key)

# --- Retry wrapper ---
def call_llm_with_retry(system_prompt, user_message, max_retries=3):
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_message}
                ],
                temperature=0.2
            )
            return json.loads(response.choices[0].message.content)
        except Exception as e:
            if "rate_limit" in str(e).lower() or "429" in str(e):
                wait_time = (attempt + 1) * 3
                print(f"Rate limited, waiting {wait_time}s...")
                time.sleep(wait_time)
            else:
                print(f"Non-rate-limit error: {e}")
                break
    return None

print("Setup complete. API key loaded:", api_key[:8] if api_key else "NOT FOUND")

## 39 bug validation

In [12]:
result = run_one_test(test_bugs[0], 1)


TEST #1 — Source: Python
Severity: Critical | Component: Security/Authentication
Error Type: PermissionError | Location: Cell In[88], line 2

Root Cause: The PermissionError likely occurred because the program attempted to write to a system file ('/etc/shadow') without sufficient privileges, possibly due to a misconfigured or missing access control mec
Confidence: 0.62

Recommended Fix: Run the program with elevated privileges or modify the access control mechanism to allow writing to the '/etc/shadow' file, but consider the security implications and potential risks of such an action.
References used: [{'type': 'historical_bug', 'bug_id': 'NSS-126289', 'summary': 'Unrelated SSL handshake issue, but indicates potential security/auth issues', 'match': 'unrelated', 'similarity': 0.2}, {'type': 'historical_bug', 'bug_id': 'MOZILLA.ORG_GRAVEYARD-417062', 'summary': 'Unrelated connection issue, but may hint at broader system configuration problems', 'match': 'unrelated', 'similarity': 0.1}]

In [61]:
result["hallucinated"] = "No"          # Yes / No / Unsure
result["historical_grounding"] = "Weak"   # or "No refs" / "Weak"
result["fix_correct"] = "Partial"          # Yes / Partial / No
result["notes"] = "Technically responds to the error, but recommends a risky practice (elevated privileges) rather than the safer fix (use non-system path) that the dataset expected."
test_results.append(result)

In [52]:
result = run_one_test(test_bugs[1], 1)


TEST #1 — Source: Python
Severity: Medium | Component: String Handling/Encoding
Error Type: AttributeError | Location: Cell In[62], line 3

Root Cause: The AttributeError likely occurred because the get_size() method was called on an object that does not have this method, possibly due to incorrect handling of string encoding or buffering, similar to 
Confidence: 0.72

Recommended Fix: Replace the get_size() method call with the correct method to get the length of a string in Python, which is len().
References used: [{'type': 'historical_bug', 'bug_id': 'GROOVY-920', 'summary': 'Missing size() method on StringBuffer', 'match': 'similar', 'similarity': 0.6}, {'type': 'historical_bug', 'bug_id': 'BEEHIVE-1032', 'summary': 'Encoding handling issue in TestRecorder', 'match': 'similar', 'similarity': 0.4}]

--- Expected (from dataset) ---
Expected error type: AttributeError
Expected fix: Use len() to obtain a string's character length instead of calling a non-existent method.


In [62]:
result["hallucinated"] = "No"          # Yes / No / Unsure
result["historical_grounding"] = "Weak"   # or "No refs" / "Weak"
result["fix_correct"] = "yes"          # Yes / Partial / No
result["notes"] = "Clean, precise match to expected fix."
test_results.append(result)

In [53]:
result = run_one_test(test_bugs[2], 1)


TEST #1 — Source: Python
Severity: Medium | Component: Data Parsing
Error Type: ValueError | Location: database_375.py: line 2

Root Cause: The ValueError likely occurred because the input string 'invalid_number' could not be converted to a float, possibly due to a formatting issue or an unexpected value, similar to patterns seen in histo
Confidence: 0.62

Recommended Fix: Add input validation to ensure that the string can be converted to a float before attempting the conversion, and handle the case where the conversion fails.
References used: [{'type': 'historical_bug', 'bug_id': 'DERBY-452', 'summary': 'Import/export fails for DECIMAL type', 'match': 'similar', 'similarity': 0.4}, {'type': 'historical_bug', 'bug_id': 'Z_ARCHIVED-384101', 'summary': 'NullValueException for invalid JavaScript value', 'match': 'similar', 'similarity': 0.3}]

--- Expected (from dataset) ---
Expected error type: ValueError
Expected fix: Verify string content or intercept numeric parsing issues with an ex

In [63]:
result["hallucinated"] = "No"          # Yes / No / Unsure
result["historical_grounding"] = "Weak"   # or "No refs" / "Weak"
result["fix_correct"] = "yes"          # Yes / Partial / No
result["notes"] = "Generic but conceptually correct, matches expected direction."
test_results.append(result)

In [54]:
# for i in range(3,39):
#     result = run_one_test(test_bugs[i], 1)


TEST #1 — Source: Java
Severity: Medium | Component: Authentication
Error Type: NullPointerException | Location: Api_362.java:80

Root Cause: The NullPointerException likely occurred because an object was accessed before being properly initialized, a pattern consistent with similar historical NPEs caused by missing configuration or setup st
Confidence: 0.78

Recommended Fix: Initialize the 'metricsData_362' variable before calling the 'indexOf' method, or add a null check to prevent the NullPointerException.
References used: [{'type': 'historical_bug', 'bug_id': 'GMF-RUNTIME-282380', 'summary': 'NPE from missing null check before method call', 'match': 'similar', 'similarity': 0.8}, {'type': 'historical_bug', 'bug_id': 'JETTY-388073', 'summary': 'NPE from null session id in HashSessionIdManager', 'match': 'similar', 'similarity': 0.7}, {'type': 'historical_bug', 'bug_id': 'AXIS-1261', 'summary': 'NPE related to engineConfigFactory and concurrent requests', 'match': 'similar', 'similar

In [71]:
# import time

# for i in range(8, 14):   # first batch: 6 bugs
#     result = run_one_test(test_bugs[i], i + 1)
#     time.sleep(5)


TEST #9 — Source: Rust
Severity: Medium | Component: Array/Index Handling
Error Type: Index Out of Bounds | Location: config_375.rs:36:20

Root Cause: The Index Out of Bounds error likely occurred because the index 378 is beyond the bounds of the array profile_375, possibly due to an incorrect array size or an off-by-one error, similar to the patter
Confidence: 0.72

Recommended Fix: Add a bounds check before accessing the array index, and handle the case where the index is out of bounds.
References used: [{'type': 'historical_bug', 'bug_id': 'MNG-1819', 'summary': 'StringIndexOutOfBoundsException due to incorrect indexing', 'match': 'similar', 'similarity': 0.6}, {'type': 'historical_bug', 'bug_id': 'PLATFORM-46156', 'summary': 'Crash due to KERN_PROTECTION_FAILURE, possible memory access issue', 'match': 'unrelated', 'similarity': 0.2}]

--- Expected (from dataset) ---
Expected error type: Rust Runtime Panic: Index Out of Bounds
Expected fix: Leverage boundary conditional guards or 

In [74]:
# import time
# for i in range(11, 15):
#     result = run_one_test(test_bugs[i], i + 1)
#     time.sleep(8)


TEST #12 — Source: Go
Severity: Low | Component: Code Quality/Compiler Warnings
Error Type: Unknown | Location: Not determined

Root Cause: Unable to determine root cause due to a processing error.
Confidence: 0.0

Recommended Fix: Unable to generate a recommendation due to a processing error.
References used: []

--- Expected (from dataset) ---
Expected error type: Go Compiler Error: Unused Import
Expected fix: Remove the unused package import from the import statement block.

TEST #13 — Source: TypeScript
Severity: Medium | Component: Type Checking/Compilation
Error Type: Unknown | Location: Not determined

Root Cause: Unable to determine root cause due to a processing error.
Confidence: 0.0

Recommended Fix: Unable to generate a recommendation due to a processing error.
References used: []

--- Expected (from dataset) ---
Expected error type: TypeScript Error: TS2339 (Property Does Not Exist)
Expected fix: Refactor 'dbCursor_362' to use a strongly typed Interface containing the 'va

In [79]:
# for i in [11, 12, 13, 14]:
#     result = run_one_test(test_bugs[i], i + 1)
#     time.sleep(10)


TEST #12 — Source: Go
Severity: Low | Component: Code Quality/Compiler Warnings
Error Type: Unknown | Location: Not determined

Root Cause: Unable to determine root cause due to a processing error.
Confidence: 0.0

Recommended Fix: Unable to generate a recommendation due to a processing error.
References used: []

--- Expected (from dataset) ---
Expected error type: Go Compiler Error: Unused Import
Expected fix: Remove the unused package import from the import statement block.

TEST #13 — Source: TypeScript
Severity: Medium | Component: TypeScript Compiler
Error Type: Unknown | Location: Not determined

Root Cause: Unable to determine root cause due to a processing error.
Confidence: 0.0

Recommended Fix: Unable to generate a recommendation due to a processing error.
References used: []

--- Expected (from dataset) ---
Expected error type: TypeScript Error: TS2339 (Property Does Not Exist)
Expected fix: Refactor 'dbCursor_362' to use a strongly typed Interface containing the 'value' k

In [ ]:
result = run_one_test(test_bugs[11], 11 + 1)
    # time.sleep(10)

In [64]:
result = run_one_test(test_bugs[3], 1)


TEST #1 — Source: Java
Severity: Medium | Component: SessionManager
Error Type: NullPointerException | Location: Api_362.java:80

Root Cause: The NullPointerException likely occurred because an object was accessed before being properly initialized, possibly due to a missing null check or an uninitialized variable, similar to patterns seen i
Confidence: 0.81

Recommended Fix: Unable to generate a recommendation due to a processing error.
References used: []

--- Expected (from dataset) ---
Expected error type: Java Runtime Exception: NullPointerException
Expected fix: Perform non-null conditional check sweeps on variable string 'metricsData_362' before extracting data features.


In [65]:
result["hallucinated"] = "No"          # Yes / No / Unsure
result["historical_grounding"] = "Strong"   # or "No refs" / "Weak"
result["fix_correct"] = "yes"          # Yes / Partial / No
result["notes"] = "Strong result: correctly identifies the null-variable problem, gives the expected null-check fix, and has strong historical support."
test_results.append(result)

In [66]:
result = run_one_test(test_bugs[4], 1)


TEST #1 — Source: Java
Severity: Medium | Component: Compiler/TypeChecker
Error Type: Incompatible Types Error | Location: Database_74.java:3

Root Cause: Unable to determine root cause due to a processing error.
Confidence: 0.0

Recommended Fix: Unable to generate a recommendation due to a processing error.
References used: []

--- Expected (from dataset) ---
Expected error type: Java Compiler Error: Incompatible Types
Expected fix: Match the target variable data type or apply explicit parsing methods like Integer.parseInt().


In [67]:
result["hallucinated"] = "No"          # Yes / No / Unsure
result["historical_grounding"] = "Weak"   # or "No refs" / "Weak"
result["fix_correct"] = "yes"          # Yes / Partial / No
result["notes"] = "Correct and practical fix that matches the expected type-matching/parsing solution, although historical grounding is only weak."
test_results.append(result)

In [68]:
result = run_one_test(test_bugs[5], 1)


TEST #1 — Source: Java
Severity: Medium | Component: DatabaseConnector
Error Type: Unknown | Location: Not determined

Root Cause: Unable to determine root cause due to a processing error.
Confidence: 0.0

Recommended Fix: Unable to generate a recommendation due to a processing error.
References used: []

--- Expected (from dataset) ---
Expected error type: Java Runtime Exception: ClassCastException
Expected fix: Use safety validation features via the 'instanceof' keyword modifier prior to running narrowing object references casts.


In [69]:
result["hallucinated"] = "No"          # Yes / No / Unsure
result["historical_grounding"] = "Strong"   # or "No refs" / "Weak"
result["fix_correct"] = "yes"          # Yes / Partial / No
result["notes"] = "Strong result: correctly identifies the null-variable problem, gives the expected null-check fix, and has strong historical support."
test_results.append(result)

In [70]:
result = run_one_test(test_bugs[8], 9)


TEST #9 — Source: Rust
Severity: Medium | Component: Array/Indexing
Error Type: Index Out of Bounds | Location: config_375.rs:36:20

Root Cause: The Index Out of Bounds error likely occurred because the index 378 is greater than or equal to the length of the profile_375 array, similar to the pattern seen in the StringIndexOutOfBoundsException 
Confidence: 0.72

Recommended Fix: Add a bounds check before accessing the array index, and handle the case where the index is out of bounds.
References used: [{'type': 'historical_bug', 'bug_id': 'MNG-1819', 'summary': 'Out of bounds index caused StringIndexOutOfBoundsException', 'match': 'similar', 'similarity': 0.8}, {'type': 'historical_bug', 'bug_id': 'PLATFORM-46156', 'summary': 'Unrelated Eclipse crash, but possible indexing issue', 'match': 'unrelated', 'similarity': 0.2}]

--- Expected (from dataset) ---
Expected error type: Rust Runtime Panic: Index Out of Bounds
Expected fix: Leverage boundary conditional guards or the safe '.get()' i

In [6]:
result = run_one_test(test_bugs[10], 11)


TEST #11 — Source: Go
Severity: Low | Component: Code Quality/Compiler Warnings
Error Type: Declared and Not Used | Location: ./auth_74.go:57:5

Root Cause: The 'Declared and Not Used' error likely occurred because a variable, payload_74, was declared but not utilized in the code, possibly due to a leftover from previous development or a mistake in variab
Confidence: 0.58

Recommended Fix: Remove the unused variable 'payload_74' from the code to eliminate the 'Declared and Not Used' error.
References used: [{'type': 'historical_bug', 'bug_id': 'GROOVY-365', 'summary': 'Variable scope issue, somewhat related to unused variables', 'match': 'somewhat related', 'similarity': 0.4}, {'type': 'historical_bug', 'bug_id': 'GROOVY-995', 'summary': 'Incorrect function usage, possible code quality issue', 'match': 'loosely related', 'similarity': 0.3}]

--- Expected (from dataset) ---
Expected error type: Go Compiler Error: Unused Variable
Expected fix: Remove variable declaration 'payload_74' or

In [26]:
result = run_one_test(test_bugs[11], 12)


TEST #12 — Source: Go
Severity: Low | Component: Code Quality/Compiler Warnings
Error Type: Unused Import | Location: ./handlers_375.go:4:5

Root Cause: Unable to determine root cause due to a processing error.
Confidence: 0.0

Recommended Fix: Unable to generate a recommendation due to a processing error.
References used: []

--- Expected (from dataset) ---
Expected error type: Go Compiler Error: Unused Import
Expected fix: Remove the unused package import from the import statement block.


In [13]:
result = run_one_test(test_bugs[12], 13)


TEST #13 — Source: TypeScript
Severity: Medium | Component: TypeScript Compiler/Utils
Error Type: TypeScript Error: TS2339 (Property Does Not Exist on Type) | Location: utils_362.ts:34:13

Root Cause: The TypeScript Error TS2339 is likely caused by an incorrect or missing type definition for the 'dbCursor_362' object, resulting in the property 'value' not being recognized as existing on that type, 
Confidence: 0.72

Recommended Fix: Update the type definition for 'dbCursor_362' to include the 'value' property, or use a type assertion to inform TypeScript that 'dbCursor_362' has a 'value' property.
References used: [{'type': 'historical_bug', 'bug_id': 'CORE-509184', 'summary': 'JSON serialization issue with property access', 'match': 'similar', 'similarity': 0.6}, {'type': 'historical_bug', 'bug_id': 'JSDT-507518', 'summary': 'Formatter issue with nested brackets, potentially related to type checking', 'match': 'loosely related', 'similarity': 0.4}]

--- Expected (from dataset) ---
Ex

In [84]:
result = run_one_test(test_bugs[13], 14) 


TEST #14 — Source: TypeScript
Severity: Medium | Component: Database
Error Type: TypeScript Error: TS2564 (Property Not Definitely Assigned) | Location: types_74.ts:2:5

Root Cause: The TypeScript Error TS2564 is likely caused by the property 'connectionString' not being definitely assigned in the Database class, possibly due to a missing or incorrect initialization in the constr
Confidence: 0.72

Recommended Fix: Initialize the 'connectionString' property in the Database class constructor or provide a default value to ensure it is definitely assigned.
References used: [{'type': 'historical_bug', 'bug_id': 'DIRSERVER-482', 'summary': 'Database and ContextPartition refactoring issue', 'match': 'similar', 'similarity': 0.6}, {'type': 'historical_bug', 'bug_id': 'FTPSERVER-22', 'summary': 'Constructor properties configuration issue', 'match': 'similar', 'similarity': 0.7}, {'type': 'historical_bug', 'bug_id': 'XALANC-604', 'summary': 'Initialization issue with built-in types', 'match': '

In [85]:
result = run_one_test(test_bugs[14], 15)


TEST #15 — Source: TypeScript
Severity: Medium | Component: UI/Rendering
Error Type: TypeScript Error: TS2531 (Object is Possibly 'null') | Location: config_375.ts:2:1

Root Cause: The TypeScript Error TS2531 (Object is Possibly 'null') likely occurred because the payload_375 object was not properly checked for null before adding an event listener, similar to patterns seen in hi
Confidence: 0.72

Recommended Fix: Add a null check for the payload_375 object before adding an event listener to ensure it is not null or undefined.
References used: [{'type': 'historical_bug', 'bug_id': 'SEAMONKEY-305158', 'summary': 'Content area click handler broke due to null object', 'match': 'similar', 'similarity': 0.8}, {'type': 'historical_bug', 'bug_id': 'CALENDAR-247865', 'summary': 'Undefined property reference caused error, similar to possible null object issue', 'match': 'similar', 'similarity': 0.7}]

--- Expected (from dataset) ---
Expected error type: TypeScript Error: TS2531 (Object is Possi

In [86]:
result = run_one_test(test_bugs[15], 16)


TEST #16 — Source: PHP
Severity: Critical | Component: JSON Encoding/Server-Side Logic
Error Type: Fatal Error: Call to Undefined Function | Location: /var/www/html/Logger_362.php:42

Root Cause: The Fatal Error: Call to Undefined Function is likely caused by a typo or incorrect function name in the PHP code, possibly related to JSON encoding or server-side logic, similar to patterns seen in h
Confidence: 0.72

Recommended Fix: Correct the typo in the function name 'json_ecnode_typo_$362' to the correct function name, likely 'json_encode'.
References used: [{'type': 'historical_bug', 'bug_id': 'PDT-151264', 'summary': 'Incorrect curly bracket sign in PHP code', 'match': 'similar', 'similarity': 0.6}, {'type': 'historical_bug', 'bug_id': 'ECLIPSELINK-419072', 'summary': 'Incorrect character marshalling to JSON OutputStream', 'match': 'similar', 'similarity': 0.4}]

--- Expected (from dataset) ---
Expected error type: PHP Fatal Error: Call to Undefined Function
Expected fix: Correct the

In [7]:
result = run_one_test(test_bugs[16], 17)


TEST #17 — Source: PHP
Non-rate-limit error: Invalid \escape: line 1 column 123 (char 122)
Severity: Medium | Component: Tax Calculation Function
Error Type: Unknown | Location: Not determined

Root Cause: Unable to determine root cause due to a processing error.
Confidence: 0.0

Recommended Fix: Unable to generate a recommendation due to a processing error.
References used: []

--- Expected (from dataset) ---
Expected error type: PHP TypeError
Expected fix: Pass matching variable types as declared in the function's strict type signature.


In [89]:
result = run_one_test(test_bugs[17], 18)


TEST #18 — Source: PHP
Severity: Medium | Component: Registration
Error Type: ArgumentCountError | Location: /var/www/html/User_375.php:97

Root Cause: Unable to determine root cause due to a processing error.
Confidence: 0.0

Recommended Fix: Unable to generate a recommendation due to a processing error.
References used: []

--- Expected (from dataset) ---
Expected error type: PHP ArgumentCountError
Expected fix: Provide all mandatory arguments required by the function signature or define default values for parameters.


In [14]:
result = run_one_test(test_bugs[18], 19)


TEST #19 — Source: CSS
Severity: Medium | Component: UI/Rendering
Error Type: PostCSS Parser Error: Unclosed Block | Location: navigation_362.css:46:1

Root Cause: The PostCSS Parser Error is likely caused by an unclosed block in the CSS source, specifically before the '.element-sibling_362' declaration, which is a common cause of parser errors in CSS.
Confidence: 0.92

Recommended Fix: Close the CSS declaration block before '.element-sibling_362' to fix the PostCSS Parser Error.
References used: [{'type': 'historical_bug', 'bug_id': 'CORE-261073', 'summary': 'Parser error due to bad selector', 'match': 'similar', 'similarity': 0.6}, {'type': 'historical_bug', 'bug_id': 'CORE-265897', 'summary': 'Crash caused by CSS rule issue', 'match': 'similar', 'similarity': 0.5}]

--- Expected (from dataset) ---
Expected error type: PostCSS Parser Error: Unclosed Block
Expected fix: Introduce a closing curly brace '}' to isolate ruleset '.btn-action_362' before defining succeeding elements.


In [15]:
result = run_one_test(test_bugs[19], 20)


TEST #20 — Source: CSS
Severity: Low | Component: UI/Rendering
Error Type: Unknown | Location: Not determined

Root Cause: Unable to determine root cause due to a processing error.
Confidence: 0.0

Recommended Fix: Unable to generate a recommendation due to a processing error.
References used: []

--- Expected (from dataset) ---
Expected error type: Stylelint Error: Invalid Hex Color
Expected fix: Correct the hex color containing non-hexadecimal character 'g'.


In [16]:
result = run_one_test(test_bugs[20], 21)


TEST #21 — Source: CSS
Severity: Low | Component: UI/Rendering
Error Type: Unknown | Location: Not determined

Root Cause: Unable to determine root cause due to a processing error.
Confidence: 0.0

Recommended Fix: Unable to generate a recommendation due to a processing error.
References used: []

--- Expected (from dataset) ---
Expected error type: Stylelint Error: Duplicate Property
Expected fix: Remove the duplicate 'display' layout parameter to avoid cascading overrides.


In [17]:
result = run_one_test(test_bugs[21], 22)


TEST #22 — Source: HTML
Severity: Low | Component: UI/Rendering
Error Type: Unknown | Location: Not determined

Root Cause: Unable to determine root cause due to a processing error.
Confidence: 0.0

Recommended Fix: Unable to generate a recommendation due to a processing error.
References used: []

--- Expected (from dataset) ---
Expected error type: W3C Validator Error: Missing Required Attribute
Expected fix: Add an 'alt' attribute describing the graphic for Web Accessibility (A11Y).


In [18]:
result = run_one_test(test_bugs[22], 23)


TEST #23 — Source: HTML
Severity: Medium | Component: UI/Rendering
Error Type: Unknown | Location: Not determined

Root Cause: Unable to determine root cause due to a processing error.
Confidence: 0.0

Recommended Fix: Unable to generate a recommendation due to a processing error.
References used: []

--- Expected (from dataset) ---
Expected error type: W3C Validator Error: Element Not Allowed Here
Expected fix: Wrap the child content of a <ul> block inside a valid <li> element instead of a <div>.


In [19]:
result = run_one_test(test_bugs[23], 24)


TEST #24 — Source: HTML
Non-rate-limit error: Connection error.
Severity: Medium | Component: Unknown
Error Type: Unknown | Location: Not determined

Root Cause: Unable to determine root cause due to a processing error.
Confidence: 0.0

Recommended Fix: Unable to generate a recommendation due to a processing error.
References used: []

--- Expected (from dataset) ---
Expected error type: W3C Validator Error: Duplicate Attribute
Expected fix: Combine separate CSS class references into a single attribute separated by spaces.


In [20]:
result = run_one_test(test_bugs[24], 25)


TEST #25 — Source: C++
Severity: Medium | Component: Data Structures/STL
Error Type: Unknown | Location: Not determined

Root Cause: Unable to determine root cause due to a processing error.
Confidence: 0.0

Recommended Fix: Unable to generate a recommendation due to a processing error.
References used: []

--- Expected (from dataset) ---
Expected error type: g++ Compiler Error: Template Mismatch
Expected fix: Instantiate a valid std::pair or use braces/insert_or_assign to add map components.


In [21]:
result = run_one_test(test_bugs[25], 26)


TEST #26 — Source: C++
Severity: Medium | Component: Unknown
Error Type: Unknown | Location: Not determined

Root Cause: Unable to determine root cause due to a processing error.
Confidence: 0.0

Recommended Fix: Unable to generate a recommendation due to a processing error.
References used: []

--- Expected (from dataset) ---
Expected error type: std::bad_alloc Exception
Expected fix: Intercept allocation exceptions with try-catch blocks or use std::nothrow to safely receive null layouts.


In [22]:
result = run_one_test(test_bugs[26], 27)


TEST #27 — Source: C++
Severity: Medium | Component: Unknown
Error Type: Unknown | Location: Not determined

Root Cause: Unable to determine root cause due to a processing error.
Confidence: 0.0

Recommended Fix: Unable to generate a recommendation due to a processing error.
References used: []

--- Expected (from dataset) ---
Expected error type: g++ Compiler Warning: Safe Comparison Mismatch (Signed/Unsigned)
Expected fix: Iterate using matching 'size_t' indices or loop references to prevent unsigned integer overflows.


In [23]:
result = run_one_test(test_bugs[27], 28)


TEST #28 — Source: C
Severity: Medium | Component: Compiler/Parser
Error Type: Unknown | Location: Not determined

Root Cause: Unable to determine root cause due to a processing error.
Confidence: 0.0

Recommended Fix: Unable to generate a recommendation due to a processing error.
References used: []

--- Expected (from dataset) ---
Expected error type: GCC Compiler Error: Syntax Error
Expected fix: Include a semi-colon identifier directly after the variable definition for 'matrix_362'.


In [ ]:
result = run_one_test(test_bugs[28], 29)

In [ ]:
result = run_one_test(test_bugs[29], 30)

In [ ]:
result = run_one_test(test_bugs[30], 31)

In [ ]:
result = run_one_test(test_bugs[31], 32)

In [ ]:
result = run_one_test(test_bugs[32], 33)

In [ ]:
result = run_one_test(test_bugs[33], 34)

## TEST REPORT 

**Methodology**
- I have  sampled 39 bugs total, reproducibly (fixed random seed = 42), across 14 sources — 11 language-specific synthetic datasets (Python, Java, Rust, Go, TypeScript, PHP, CSS, HTML, C++, C, JavaScript, 3 bugs each = 33) plus 3 real-world historical datasets (Mozilla, Eclipse, Apache — 2 each = 6). Each bug was submitted through the  multi-agent pipeline (Triage → Log Analysis → Root Cause → Remediation), to test the system end-to-end exactly as a real user would experience it.

**Judgment Criteria**

- Each completed test was judged on three criteria/conditions:

- Hallucination — did the recommended fix logically  from the actual input (if bug was same or similiar  as previous so was that taken from from KB to understand and retrived if it was same ), or did it invent something unconnected to the real error?
- Historical grounding — whether it supports historical evidence or not , using the system's own similarity scores: Strong (≥0.7), Weak (0.4–0.69), None (<0.4 or empty).
- Fix correctness — did the recommended fix address the same underlying problem as the dataset's expected fix (Yes / Partial / No)?

**Results Summary**

   | SNo |   Outcome  | count |
   |:------|:----:|:------|
   |1|Fully completed (all 4 agents produced real output)|16|
   |2|Partially completed (Triage/Log Analysis succeeded, Root Cause and/or Remediation failed mid-pipeline)|2|
   |3|Not completed (blocked before producing output)|21|


- Of the 15 fully completed tests, spanning Python, Java, Rust, Go, TypeScript, PHP, and CSS:

- 0 of 15 showed evidence of hallucination — every recommended fix logically followed from the actual input text.
- Historical grounding: 2 Strong, 11 Weak, 2 None/unrelated.
- Fix correctness: 11 Yes, 4 Partial, 0 No.

**Key Finding: Consistently Weak-to-Moderate Historical Grounding**

- Across nearly all completed tests, similarity scores for historical references clustered in the 0.4–0.7 range, rarely exceeding 0.7. This suggests the knowledge base — while broad (56,000+ entries) — doesn't always contain closely-matching precedents for arbitrary, newly-introduced bug types, particularly ones from languages underrepresented in the original Mozilla/Eclipse/Apache source data (e.g., Rust, Go, TypeScript). This is an honest limitation of coverage, not of the retrieval mechanism itself, which was mathematically correct in every case.

**Operational Limitation: Daily API Quota**  

- Testing was not done completly due to LLM provider's daily token quota (Groq on-demand tier, 100,000 tokens/day). Of 39 planned test cases, 16 received complete pipeline output and 2 partially completed before quota exhaustion; the remaining 21 could not be completed within the testing window. This is disclosed as a real operational constraint of building on a metered LLM API, not a defect in the pipeline itself — every completed test produced structurally valid, non-hallucinated output. This finding is itself relevant to the project's Milestone 4 requirement to validate system behavior under realistic constraints: a production deployment would need either a paid API tier, request batching/caching strategies, or a fallback model to sustain higher testing/usage volume

**Documented Edge Cases from Earlier Testing**

- Compound multi-bug submissions causing error_type to become a comma-separated list (found during Analytics testing)
- Semantic clustering catching label inconsistencies (Cosmetic Typo vs Typographical Error) that exact-match grouping missed

In [8]:
"IF BUG WAS AME AS PREVIOUS SO WAS FIX TAKEN FROM".lower()

'if bug was ame as previous so was fix taken from'

## Testing 

In [12]:
def subb(a, b):
    return a - b
print(subb(3, 4))

-1


In [9]:
if 8>9 
    print("correct")
else:
 print('wrong')

SyntaxError: expected ':' (3058357825.py, line 1)

In [1]:
import pandas as pd
import numpy as np

metadata = pd.read_csv("../src/chunks_metadata.csv")
embeddings = np.load("../src/embeddings_real.npy")

print("Metadata rows:", len(metadata))
print("Embeddings shape:", embeddings.shape)

Metadata rows: 56867
Embeddings shape: (56867, 384)


In [2]:
live_entries = metadata[metadata['source_dataset'] == 'live_submissions']
print(f"Live-resolved bugs added to KB: {len(live_entries)}")
print(live_entries[['bug_id', 'title', 'resolution', 'status']].to_string())

Live-resolved bugs added to KB: 5
             bug_id               title                                                                                                           resolution    status
56862  BUG-d5b55cac           NameError                       Define the function 'adddd' before it is called, or correct the function name if it is a typo.  resolved
56863  BUG-a6564d59           NameError    Correct the typo in the function name from 'subbb' to 'subb', or define the function 'subbb' before it is called.  resolved
56864  BUG-11babb9c         SyntaxError                                                    Add a colon at the end of the if statement to correct the syntax.  resolved
56865  BUG-5809b03c  Cannot Find Symbol                                                            Correct the typo in the System class from 'Out' to 'out'.  resolved
56866  BUG-3e2e10ec           TypeError  Correct the typo in the console.Log function call to console.log, ensuring proper JavaScri